In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms
from torchvision.datasets import Food101
import matplotlib.pyplot as plt


# ==========================================
# 1. CONFIG
# ==========================================
BATCH_SIZE = 32
NUM_CLASSES = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Do testów ustawiamy:
EPOCHS_HEAD = 5       # feature extraction
EPOCHS_FINETUNE = 5   # fine-tuning
EPOCHS_SCRATCH = 5    # scratch
EARLY_STOPPING_PATIENCE = 2


# ==========================================
# 2. Food101 DATALOADERS
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

# Pobieranie datasetu
full_train = Food101(root="./data", split="train", download=True, transform=train_transform)
test_set = Food101(root="./data", split="test", download=True, transform=test_transform)

# Train/Val split
train_size = int(0.8 * len(full_train))
val_size   = len(full_train) - train_size

train_set, val_set = random_split(full_train, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)


# ==========================================
# 3. UNIVERSAL TRAINING FUNCTION + EARLY STOPPING
# ==========================================
def train_model(model, criterion, optimizer, train_loader, val_loader,
                epochs, name="Model", patience=2):

    train_losses, val_losses, val_accs = [], [], []
    best_loss = float("inf")
    no_improve = 0
    best_state = None

    model.to(DEVICE)

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        # ===== VALIDATION =====
        model.eval()
        val_loss = 0
        correct, total = 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss /= len(val_loader)
        acc = correct / total

        val_losses.append(val_loss)
        val_accs.append(acc)

        print(f"[{name}] Epoch {epoch+1}/{epochs} "
              f"| Train Loss: {train_loss:.4f} "
              f"| Val Loss: {val_loss:.4f} "
              f"| Val Acc: {acc:.4f}")

        # EARLY STOPPING
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict().copy()
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # restore best model
    model.load_state_dict(best_state)

    return train_losses, val_losses, val_accs


# ==========================================
# 4. WARIANT A — FEATURE EXTRACTION
# ==========================================
feature_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# zamrażamy wszystkie warstwy
for p in feature_model.parameters():
    p.requires_grad = False

# nowa głowica
feature_model.fc = nn.Linear(feature_model.fc.in_features, NUM_CLASSES)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(feature_model.fc.parameters(), lr=1e-3)

print("\n=== WARIANT A: Feature Extraction ===")
A_train, A_val, A_acc = train_model(
    feature_model, criterion, optimizer,
    train_loader, val_loader,
    epochs=EPOCHS_HEAD, name="FeatureExtraction",
    patience=EARLY_STOPPING_PATIENCE
)


# ==========================================
# 5. WARIANT B — FINE TUNING
# ==========================================
finetune_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# zamrażamy wszystko
for p in finetune_model.parameters():
    p.requires_grad = False

# odmrażamy ostatni blok ResNet (layer4)
for p in finetune_model.layer4.parameters():
    p.requires_grad = True

finetune_model.fc = nn.Linear(finetune_model.fc.in_features, NUM_CLASSES)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, finetune_model.parameters()),
    lr=1e-4
)

print("\n=== WARIANT B: Fine-Tuning Layer4 ===")
B_train, B_val, B_acc = train_model(
    finetune_model, criterion, optimizer,
    train_loader, val_loader,
    epochs=EPOCHS_FINETUNE, name="FineTuning",
    patience=EARLY_STOPPING_PATIENCE
)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28,256), nn.ReLU(),
            nn.Linear(256,num_classes)
        )

    def forward(self,x):
        x = self.features(x)
        return self.classifier(x)

scratch = SimpleCNN(NUM_CLASSES)
optimizer = optim.Adam(scratch.parameters(), lr=1e-3)

print("\n=== MODEL SCRATCH ===")
S_train, S_val, S_acc = train_model(
    scratch, criterion, optimizer,
    train_loader, val_loader,
    epochs=EPOCHS_SCRATCH, name="Scratch",
    patience=EARLY_STOPPING_PATIENCE
)


# ==========================================
# 7. WYKRESY
# ==========================================

plt.figure(figsize=(14,5))

# ACCURACY
plt.subplot(1,2,1)
plt.plot(A_acc, label="Feature Extraction")
plt.plot(B_acc, label="Fine-Tuning")
plt.plot(S_acc, label="Scratch")
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# LOSSES
plt.subplot(1,2,2)
plt.plot(A_val, label="Feature Extraction")
plt.plot(B_val, label="Fine-Tuning")
plt.plot(S_val, label="Scratch")
plt.title("Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()



=== WARIANT A: Feature Extraction ===


---

## 1️⃣ Import modelu + weights

```python
from torchvision.models import convnext_base, ConvNeXt_Base_Weights

model = convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1)
```

dla EfficientNet:

```python
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
```

dla ViT:

```python
from torchvision.models import vit_b_16, ViT_B_16_Weights
model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
```

---

## 2️⃣ Zamrożenie warstw (feature extraction)

```python
for param in model.parameters():
    param.requires_grad = False
```

Możesz też odmrozić część (fine-tuning).

---

## 3️⃣ Podmiana klasyfikatora (tail)

Każdy model kończy się inną warstwą:

| Model        | Ostatnia warstwa      |
| ------------ | --------------------- |
| ResNet       | `model.fc`            |
| ConvNeXt     | `model.classifier[2]` |
| EfficientNet | `model.classifier[1]` |
| ViT          | `model.heads.head`    |
| Swin         | `model.head`          |
| RegNet       | `model.fc`            |

---

### Przykład:

```python
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
```

---

## 4️⃣ Trening — identyczny jak wcześniej

```python
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
```

---

# ⭐ Przykład 1: **ConvNeXt Base**


```python
from torchvision.models import convnext_base, ConvNeXt_Base_Weights

model = convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1)

for p in model.parameters():
    p.requires_grad = False

# Zamieniamy classifier: ConvNeXt → classifier[2]
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
```

To wszystko — model gotowy do TL.

---

# ⭐ Przykład 2: **EfficientNet-B0**

```python
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze
for p in model.parameters():
    p.requires_grad = False

# EfficientNet → classifier[1]
in_feats = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_feats, NUM_CLASSES)
```

---

# ⭐ Przykład 3: **Vision Transformer ViT-B/16**

```python
from torchvision.models import vit_b_16, ViT_B_16_Weights

model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

# Freeze
for p in model.parameters():
    p.requires_grad = False

# Head zmieniamy w Transformerze
in_feats = model.heads.head.in_features
model.heads.head = nn.Linear(in_feats, NUM_CLASSES)
```

---

# ⭐ Przykład 4: **Swin Transformer V2-B**

```python
from torchvision.models import swin_v2_b, Swin_V2_B_Weights

model = swin_v2_b(weights=Swin_V2_B_Weights.IMAGENET1K_V1)

# Freeze
for p in model.parameters():
    p.requires_grad = False

# Głowica klasyfikacji
in_feats = model.head.in_features
model.head = nn.Linear(in_feats, NUM_CLASSES)
```

---

# ⭐ Przykład 5: **RegNet-Y 16GF**

```python
from torchvision.models import regnet_y_16gf, RegNet_Y_16GF_Weights

model = regnet_y_16gf(weights=RegNet_Y_16GF_Weights.IMAGENET1K_SWAG_E2E_V1)

for p in model.parameters():
    p.requires_grad = False

in_feats = model.fc.in_features
model.fc = nn.Linear(in_feats, NUM_CLASSES)
```
